Author: Eliana Nowlis
Date: 10/29/2026
Purpose: Generate graphs to be included in magazine layout

In [66]:
# Imports
import pandas as pd
import pathlib
import altair as alt
import vega_datasets

# Settings
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [67]:
# Read in data
tourism_folder = pathlib.Path("/home/elies/data_viz/data/UN_tourism_data")
file_paths = list(tourism_folder.glob("**/*.xlsx"))

data_list = []

for i, path in enumerate(file_paths):
    data_i = pd.read_excel(path, sheet_name="Data")
    data_list.append(data_i)

tourism = pd.concat(data_list, sort=False)

In [68]:
# Add regions to the data
# Source: Composition of UN Tourism regions
regions = pd.read_excel("../data/UN_Tourism_regions_composition.xlsx", header=4)
regions = regions[["UNWTO_region_label_English", "area_label_M49_EN", "area_code_M49"]]
regions = regions.rename(columns={"UNWTO_region_label_English" : "region",
                                  "area_label_M49_EN" : "reporter_area_label"}
                        )

# Add in the latitude and longitude for these countries
# Source: GitHub Gists
geometry = pd.read_csv("https://gist.githubusercontent.com/metal3d/5b925077e66194551df949de64e910f6/raw/c5f20a037409d96958553e2eb6b8251265c6fd63/country-coord.csv")
geometry = geometry.rename(columns={"Numeric code" : "area_code_M49"})
geometry = geometry[["area_code_M49", "Latitude (average)", "Longitude (average)"]]
regions = pd.merge(regions, geometry, on = "area_code_M49", how = "left")

tourism = pd.merge(tourism, regions, on = "reporter_area_label", how = "left")

In [69]:
# Set a default theme for all charts

@alt.theme.register("my_theme", enable=True)
def my_theme():
    return {
        "config" : {
            "axis": {
                "grid": False,
                "titleFontWeight" : "normal",
                "titlePadding" : 15
            },
            "view": {
                "stroke": None,
                "width" : 450
            }
        }
    }

In [70]:
# Create a standard mapping of regions to colors
region_scale = alt.Scale(
                domain = ['Africa', 'Americas', 'East Asia and the Pacific', 'Europe', 'Middle East', 'South Asia'],
                range = ['#ff7f0e', '#9467bd', '#2ca02c', '#d62728', '#1f77b4', '#e377c2']
                )

inbound_outbound_scale = alt.Scale(
                domain = ['Inbound Tourism', 'Outbound Tourism'],
                range = ['#f1e132', '#17becf']
                )

In [71]:
# Bubble map, total visitors, 2019, by country
data_inbound = tourism[
        (tourism["year"] == 2019) &
        (tourism["indicator_code"] == "INBD_TRIP_TOTL_TOTL_VSTR") &
        (tourism["region"].notna())
    ]

data_outbound = tourism[
        (tourism["year"] == 2019) &
        (tourism["indicator_code"] == "OUTB_TRIP_TOTL_TOTL_VSTR") &
        (tourism["region"].notna())
    ]

# World map basemap via vega-datasets
world_url = "https://cdn.jsdelivr.net/npm/vega-datasets@2/data/world-110m.json"
countries = alt.topo_feature(world_url, "countries")

def bubble_map(df, title):
    base_map = (
        alt.Chart(countries)
        .mark_geoshape(fill = "#E8E4DC", stroke= "#B0AFA8", strokeWidth=0.5)
    )
    
    chart = (
        alt.Chart(df)
        .mark_circle(opacity=0.65, strokeWidth=1.5, color = "#c77548")
        .encode(
            longitude="Longitude (average):Q",
            latitude="Latitude (average):Q",
            size=alt.Size(
                "value:Q",
                legend = alt.Legend(
                    title = 'Tourists (Thousands of Trips)',
                    orient = 'bottom'
                )
            )
        ).properties(
            title = {
                "text" : [
                        title,
                        "2019"
                ]
            }
        ).project(type="equirectangular")
    )

    return (base_map + chart)

# Render chart
bubble_map(data_inbound, "Inbound Tourism") | bubble_map(data_outbound, "Outbound Tourism")

alt.HConcatChart(...)

In [72]:
# Scatter plot, expenditure x employment, 2019, by country
data = tourism[
        (tourism["year"] == 2019) &
        (tourism["indicator_code"].isin(["INBD_EXPD_BPAY_TOTL_VSTR", "EMPL_TOTL_TOTL_TOTL_TOTL_PRSN"])) &
        (tourism["region"].notna())
    ]

data_wide = (
    data.pivot_table(
        index = ["reporter_area_label", "region"],
        columns = "indicator_code",
        values = "value"
    ).dropna().reset_index()
)

def scatter_1(df):
    chart = alt.Chart(df).mark_point(filled = True, size = 70, opacity = 0.7).encode(
        alt.X("INBD_EXPD_BPAY_TOTL_VSTR:Q", axis = alt.Axis(format = "$,.0f")).title("Tourist Expenditures (Millions USD)"),
        alt.Y("EMPL_TOTL_TOTL_TOTL_TOTL_PRSN:Q", axis=alt.Axis(format="~s")).title("People Employed in Tourism"),
        color = alt.Color("region:N", title = "Region", scale = region_scale),
        tooltip = alt.Tooltip("reporter_area_label", title = "")
    ).properties(
        title = {
            "text" : [
                    "Employment and Tourist Spending by Region",
                    "2019"
            ]
        }
    ).configure_axis(
        labelFontSize=14,
        titleFontSize=14
    ).configure_legend(
        titleFontSize=14,
        labelFontSize=14 
    )

    return chart

# Render chart
scatter_1(data_wide)

alt.Chart(...)

In [73]:
# Box plot, tourism GDP, 2019, by country and region
data = tourism[
        (tourism["year"] == 2019) &
        (tourism["indicator_code"] == "ST_GDP_ZS") &
        (tourism["region"].notna())
    ]

data["value"] = data["value"] / 100

def box_1(df):
    chart = alt.Chart(df).mark_boxplot(median={"color": "black", "strokeWidth": 2}).encode(
        alt.X("value:Q", axis = alt.Axis(format = "%")).title("Tourism Direct GDP as a Proportion of Total GDP (%)"),
        alt.Y("region:N").title(""),
        color = alt.Color("region:N", legend = None, scale = region_scale)
    ).properties(
            title = {
                "text" : [
                        "Tourism Proportion of GDP by Region",
                        "2019"
                ]
            }
    ).configure_axis(
        labelFontSize=14,
        titleFontSize=14
    )

    return chart

# Render chart
box_1(data)

alt.Chart(...)

In [74]:
# Line chart, total visitors, 2012-2022, by region
data = tourism[
        (tourism["year"].between(2012, 2022)) &
        (tourism["indicator_code"] == "INBD_TRIP_TOTL_TOTL_VSTR") &
        (tourism["region"].notna())
    ]

data = data.groupby("reporter_area_label").filter(
    lambda g: g["value"].notna().all()
    and g["year"].nunique() == 11
)

region_sums = data.groupby(["region", "indicator_code", "indicator_label", "year"]
                             )["value"].sum().reset_index()

region_sums = region_sums.merge((
                                    region_sums.loc[region_sums["year"] == 2012, ["region", "value"]]
                                    .rename(columns={"value": "value_2012"})
                                ),
                                on = "region"
                             )

region_sums["indexed_value"] = (
    (region_sums["value"] / region_sums["value_2012"]) * 100
)

def line_1(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:O", axis=alt.Axis(labelAngle=0)).title(""),
        alt.Y("indexed_value:Q").title("Inbound Tourists (% Change from 2012)"),
        color = alt.Color("region:N", title = "Region", scale = region_scale, legend = alt.Legend(orient = "bottom"))
    ).properties(
            title = {
        "text" : [
                "Impact of COVID on Tourism by Region",
                "Indexed to 2012"
        ]}
    )

    covid_shading = pd.DataFrame({
        "x1": [2020],
        "x2": [2021]
    })

    shading = alt.Chart(covid_shading).mark_rect(opacity=0.15, color = "grey").encode(
        x="x1:O",
        x2="x2"
    )

    label = alt.Chart(
        pd.DataFrame({
            "x": [2021],
            "y": [190],
            "label": ["COVID Lockdown"]
        })
    ).mark_text(color = "#4d4d4d", angle=270, dy=-5, fontSize = 14).encode(
        x="x:O",
        y="y",
        text="label"
    )

    return chart + shading + label

# Render chart
line_1(region_sums).configure_axis(
        labelFontSize=14,
        titleFontSize=14
    ).configure_legend(
        titleFontSize=14,
        labelFontSize=14 
    )

alt.LayerChart(...)

In [75]:
# Line chart, total visitors, 2008-2022, Ukraine
data = tourism[
        (tourism["year"].between(2008, 2022)) &
        (tourism["indicator_code"].isin(["INBD_TRIP_TOTL_TOTL_VSTR", "OUTB_TRIP_TOTL_TOTL_VSTR"])) &
        (tourism["region"].notna()) &
        (tourism["reporter_area_label"] == "Ukraine")
    ]

data = data.replace('INBD_TRIP_TOTL_TOTL_VSTR', 'Inbound Tourism')
data = data.replace('OUTB_TRIP_TOTL_TOTL_VSTR', 'Outbound Tourism')

def line_2(df):
    chart = alt.Chart(df).mark_line().encode(
        alt.X("year:O", axis=alt.Axis(labelAngle=0, values=list(range(2008, 2023, 2)))).title(""),
        alt.Y("value:Q", scale=alt.Scale(domain=[0, 40000])).title("Tourists (Thousands of Trips)"),
        color = alt.Color("indicator_code:N", title = "Direction", scale = inbound_outbound_scale)
    ).properties(title = "Impact of Conflicts on Tourism in Ukraine")

    events = pd.DataFrame({
        "x": [2014, 2020, 2022]
    })

    lines = alt.Chart(events).mark_rule(color = "grey", opacity = 0.7).encode(
        x="x:O"
    )

    labels = alt.Chart(
        pd.DataFrame({
            "x": [2014, 2020, 2022],
            "y": [35000, 35000, 35000],
            "label": ["Crimea Occupation", "Start of COVID", "Full‑Scale Invasion"]
        })
    ).mark_text(color = "#4d4d4d", angle=270, dy=-5, fontSize = 14).encode(
        x="x:O",
        y="y:Q",
        text="label"
    )

    return chart + lines + labels

# Render chart
line_2(data).configure_axis(
        labelFontSize=14,
        titleFontSize=14
    ).configure_legend(
        titleFontSize=14,
        labelFontSize=14 
    )

alt.LayerChart(...)

In [76]:
# Bar chart, average length of stay, 1999 and 2019, by region
data = tourism[
        (tourism["year"].isin([1999, 2019])) &
        (tourism["indicator_code"] == "CPTY_ACCM_HOTL_STAY_AVRG") &
        (tourism["region"].notna())
    ]

region_sums = data.groupby(["region", "indicator_code", "year"]
                             )["value"].mean().reset_index()

def bar_1(df):
    chart = alt.Chart(df).mark_bar().encode(
        alt.X("value:Q").title("Average Length of Stay (Nights)"),
        alt.Y("region:N").title(""),
        color = alt.Color("region:N", title = "Region", scale = region_scale),
        yOffset=alt.YOffset("year:N")
    ).properties(title = "Average Length of Stay by Region", width=250)
    
    return chart

# Render chart
bar_1(region_sums).configure_axis(
        labelFontSize=14,
        titleFontSize=14
    ).configure_legend(
        titleFontSize=14,
        labelFontSize=14 
    )

alt.Chart(...)

In [77]:
# Bump chart, total visitors, 2004 to 2019, all countries
data = tourism[
        (tourism["year"].between(2004, 2019)) &
        (tourism["indicator_code"] == "INBD_TRIP_TOTL_TOTL_VSTR") &
        (tourism["region"].notna())
    ]

top10 = data.groupby("reporter_area_label")["value"].sum().nlargest(10)

data = data[data["reporter_area_label"].isin(top10.index.tolist())]

def bump_1(df):
    chart = alt.Chart(df).mark_line(point=True).encode(
        alt.X("year:O", axis=alt.Axis(labelAngle=0, values=list(range(2004, 2020, 2)))).title(""),
        alt.Y("rank:O").title("Rank"),
        color = alt.Color("reporter_area_label:N", title = "", scale=alt.Scale(scheme="tableau10"))
    ).transform_window(
        rank="rank()",
        sort=[alt.SortField("value", order="descending")],
        groupby=["year"]
    ).properties(
        title = {
        "text": "Rank of Most 'Touristy' Countries",
        "subtitle": [
            "Among top 10 countries based on cumulative tourism (2004-2019)"
        ]
    })

    return chart

# Render chart
bump_1(data).configure_axis(
        labelFontSize=14,
        titleFontSize=14
    ).configure_legend(
        titleFontSize=14,
        labelFontSize=14 
    )

alt.Chart(...)